# Exploring `figure_data.pkl`

The **source of truth** for every figure: a self-documenting dict where each array carries its dimension names (`dims`) and coordinate labels (`coords`). Raw per-trial activity lives separately in `figure_data/time_resolved/` (see `explore_time_resolved.ipynb`).

In [1]:
import pickle, numpy as np
import sys
from pathlib import Path
def _find(rel):
    for c in [Path.cwd(), *Path.cwd().parents]:
        if (c / rel).exists():
            return c / rel
    raise FileNotFoundError(rel + " (run from inside the bundle)")
CODE = _find("code"); DATA = _find("figure_data"); OUT = DATA.parent / "figures"
sys.path.insert(0, str(CODE))

D = pickle.load(open(DATA / 'figure_data.pkl', 'rb'))
print('top-level keys:', list(D))
print()
print(D['description'])

top-level keys: ['description', 'model_types', 'seeds', 'stim_labels', 'n_units', 'seeds_present', 'period', 'scalars', 'tuning', 'aligned_mean', 'responsive']

Final 3s1c continuous-vigour models: per-(model_type, seed, stimulus) metrics, per-unit stim tuning, per-unit trial-aligned means, and responsiveness. Backs every figure. Raw per-trial activity is in time_resolved/.


## Metadata

In [2]:
print('model_types  :', D['model_types'])
print('seeds        :', D['seeds'][0], '..', D['seeds'][-1], f"({len(D['seeds'])})")
print('seeds_present:', D['seeds_present'])
print('stim_labels  :', D['stim_labels'])
print('n_units      :', D['n_units'])
print('period       :', D['period'])

model_types  : ['classif_rl', 'rl_only', 'classif_rl_readout_only']
seeds        : 42 .. 71 (30)
seeds_present: {'classif_rl': [42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71], 'rl_only': [42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71], 'classif_rl_readout_only': [42, 43]}
stim_labels  : ['0%', '50%', '100%']
n_units      : 128
period       : {'n_iti_pre': 3, 'stim_ts': 5, 'rew_ts': 3, 'n_align_ts': 11, 'segments': ['ITI', 'stim', 'outcome'], 'segment_bounds': [[0, 3], [3, 8], [8, 11]]}


## `scalars` — the bar-figure metrics
Arrays shaped `(model_type, seed, stim)`; `NaN` where a (type, seed) is absent.

In [3]:
S = D['scalars']
print('dims:', S['dims'])
for m, desc in S['units'].items():
    print(f'  {m:15s} {S[m].shape}  — {desc}')
print()
ti = {m: i for i, m in enumerate(D['model_types'])}
for m in D['model_types']:
    print(m, '-> mean #responsive/stim:', np.nanmean(S['n_responsive'][ti[m]], 0).round(1))

dims: ['model_type', 'seed', 'stim']
  vigour          (3, 30, 3)  — deterministic mean vigour per stimulus, in [0,1]
  pop_activity    (3, 30, 3)  — mean hidden-unit activation in the stim window
  n_responsive    (3, 30, 3)  — # units significantly responsive (paired t-test vs pre-stim baseline)
  frac_responsive (3, 30, 3)  — fraction of units significantly responsive

classif_rl -> mean #responsive/stim: [32.2 52.5 64.2]
rl_only -> mean #responsive/stim: [11.6 64.  70.9]
classif_rl_readout_only -> mean #responsive/stim: [63.5 75.5 82. ]


## `tuning` — per-unit stim-window mean  (heatmap source)

In [4]:
print(D['tuning']['dims'], D['tuning']['data'].shape)

['model_type', 'seed', 'stim', 'unit'] (3, 30, 3, 128)


## `aligned_mean` — per-unit trial-aligned mean per stimulus  (PSTH source)
Time axis is the aligned window `ITI | stim | outcome`; `period['segment_bounds']` gives the slice of each segment.

In [5]:
A = D['aligned_mean']
print(A['dims'], A['data'].shape)
print('segments:', D['period']['segments'], 'bounds:', D['period']['segment_bounds'])

['model_type', 'seed', 'unit', 'time', 'stim'] (3, 30, 128, 11, 3)
segments: ['ITI', 'stim', 'outcome'] bounds: [[0, 3], [3, 8], [8, 11]]


## `responsive` — per-unit responsiveness pattern  (defines the cell-type groups)

In [6]:
R = D['responsive']
print(R['dims'], R['data'].shape, R['data'].dtype)
# example: count exclusively-0% units in classif_rl (pooled over seeds)
pat = R['data'][ti['classif_rl']].reshape(-1, 3)
excl0 = ((pat[:, 0]) & (~pat[:, 1]) & (~pat[:, 2])).sum()
print('classif_rl exclusively-0% units (all seeds):', int(excl0))

['model_type', 'seed', 'unit', 'stim'] (3, 30, 128, 3) bool
classif_rl exclusively-0% units (all seeds): 381
